# Chapter 11 Companion Notebook: Insurance and Actuarial Modeling

This notebook reproduces every worked numerical example from Chapter 11 of *AI in Finance*: the pooling/law-of-large-numbers calculation, float and combined-ratio economics, pure-premium rating with multiplicative relativities, the one-way rating bias, the GLM-versus-tree interaction example (with a real Poisson GLM fit), the Buhlmann credibility blend, the telematics feature-to-relativity model, claims-triage logistic scores, the chain-ladder reserving triangle, the term-life net premium, the HCC risk-adjustment score with an upcoding audit z-test, the dynamic-lapse ALM rerun, the reinsurance layer and parametric basis-risk calculations, and the Solvency-II-style 99.5% VaR normal approximation.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. Risk pooling and the law of large numbers (Section 11.2.1)

In [1]:
import math
import numpy as np
from statistics import NormalDist

n, mu, sd = 10_000, 720, 3_000
pool_mean, pool_sd = n*mu, sd*math.sqrt(n)
cv_single, cv_pool = sd/mu, pool_sd/pool_mean
z = 0.10*pool_mean/pool_sd
print(f"single-policy CV = {cv_single:.0%}; pooled mean = ${pool_mean:,}, sd = ${pool_sd:,.0f}, CV = {cv_pool:.2%}")
print(f"P(loss > 110% of mean): z = {z:.1f}, p = {1-NormalDist().cdf(z):.4f}")

single-policy CV = 417%; pooled mean = $7,200,000, sd = $300,000, CV = 4.17%
P(loss > 110% of mean): z = 2.4, p = 0.0082


## 2. Float and the combined ratio (Section 11.2.2)

In [2]:
premium, expense_ratio = 100.0, 0.30
float_amt, yield_, capital = 150.0, 0.04, 50.0

for lr in (0.65, 0.73):
    combined = lr + expense_ratio
    uw = premium*(1 - combined)
    inv = float_amt*yield_
    total = uw + inv
    print(f"combined ratio {combined:.0%}: underwriting = {uw:+.0f}M, investment = {inv:.0f}M, "
          f"total = {total:.0f}M, ROE = {total/capital:.0%}")

combined ratio 95%: underwriting = +5M, investment = 6M, total = 11M, ROE = 22%
combined ratio 103%: underwriting = -3M, investment = 6M, total = 3M, ROE = 6%


## 3. Pure-premium pricing with multiplicative relativities (Section 11.2.3)

In [3]:
base_freq, base_sev = 0.08, 9_000
pure_base = base_freq*base_sev
freq = base_freq * 1.8 * 1.25          # young x urban frequency relativities
sev = base_sev * 1.10                  # urban severity relativity
pure = freq*sev
gross = pure/(1 - 0.25 - 0.05)         # expense 25%, profit 5%
print(f"base pure premium = ${pure_base:,.0f}")
print(f"young urban: frequency = {freq:.2f}, severity = ${sev:,.0f}, pure = ${pure:,.0f}, gross = ${gross:,.2f}")

base pure premium = $720
young urban: frequency = 0.18, severity = $9,900, pure = $1,782, gross = $2,545.71


## 4. Why one-way rating fails (Section 11.3.1)

In [4]:
exposure = {'young-urban':200, 'young-rural':50, 'exp-urban':300, 'exp-rural':450}
claims   = {'young-urban':54,  'young-rural':9,  'exp-urban':45,  'exp-rural':45}

f_young = (claims['young-urban']+claims['young-rural'])/(exposure['young-urban']+exposure['young-rural'])
f_exp   = (claims['exp-urban']+claims['exp-rural'])/(exposure['exp-urban']+exposure['exp-rural'])
f_urban = (claims['young-urban']+claims['exp-urban'])/(exposure['young-urban']+exposure['exp-urban'])
f_rural = (claims['young-rural']+claims['exp-rural'])/(exposure['young-rural']+exposure['exp-rural'])

oneway_age, oneway_terr = f_young/f_exp, f_urban/f_rural
print(f"one-way age relativity: {oneway_age:.2f} (true 1.8); territory: {oneway_terr:.3f} (true 1.5)")
print(f"one-way young-urban charge: {oneway_age*oneway_terr:.2f}x base vs true 2.70x "
      f"-> overcharge {(oneway_age*oneway_terr/2.70-1):.0%}")

one-way age relativity: 2.10 (true 1.8); territory: 1.833 (true 1.5)
one-way young-urban charge: 3.85x base vs true 2.70x -> overcharge 43%


## 5. Buhlmann credibility (Section 11.3.2)

In [5]:
n_years, k = 3, 12
Z = n_years/(n_years + k)
f_own, f_class = 1/3, 0.10
blend = Z*f_own + (1-Z)*f_class
print(f"Z = {Z:.2f}; blended frequency = {blend:.4f} (own {f_own:.3f}, class {f_class:.3f})")

Z = 0.20; blended frequency = 0.1467 (own 0.333, class 0.100)


## 6. The interaction a GLM cannot price (Section 11.3.3)

In [6]:
import statsmodels.api as sm

# young-urban cell now carries a true interaction: frequency 0.32 (64 claims on 200)
exposure_v = np.array([200, 50, 300, 450], dtype=float)
claims_v   = np.array([64, 9, 45, 45], dtype=float)
young      = np.array([1, 1, 0, 0])
urban      = np.array([1, 0, 1, 0])

X = sm.add_constant(np.column_stack([young, urban]))
glm = sm.GLM(claims_v, X, family=sm.families.Poisson(), offset=np.log(exposure_v)).fit()
b0, b_young, b_urban = glm.params
print(f"fitted base = {math.exp(b0):.4f}, young relativity = {math.exp(b_young):.4f}, "
      f"urban relativity = {math.exp(b_urban):.4f}")

pred = np.exp(b0 + b_young*young + b_urban*urban)
actual = claims_v/exposure_v
for name, p, a in zip(['young-urban','young-rural','exp-urban','exp-rural'], pred, actual):
    print(f"  {name:12s}: GLM {p:.4f} vs actual {a:.4f} -> error {(p/a-1):+.1%}")
print("(a depth-two tree splits into the four cells and prices each exactly)")

fitted base = 0.0977, young relativity = 2.0523, urban relativity = 1.5701
  young-urban : GLM 0.3149 vs actual 0.3200 -> error -1.6%
  young-rural : GLM 0.2005 vs actual 0.1800 -> error +11.4%
  exp-urban   : GLM 0.1534 vs actual 0.1500 -> error +2.3%
  exp-rural   : GLM 0.0977 vs actual 0.1000 -> error -2.3%
(a depth-two tree splits into the four cells and prices each exactly)


## 7. Telematics: feature to relativity (Section 11.3.4)

In [7]:
beta_brake = -0.163          # fitted coefficient per harsh-braking event per 100 miles
delta_h = 2                  # driver brakes harshly 2 fewer times per 100 miles than cell average
multiplier = math.exp(beta_brake*delta_h)
relativity = 1.8*multiplier
freq_t = 0.08*round(relativity, 2)*1.25
pure_t = freq_t*9_900
gross_t = pure_t/0.70
saving = 2545.71 - gross_t
print(f"telematics multiplier = e^({beta_brake*delta_h:.3f}) = {multiplier:.3f}; relativity 1.8 -> {relativity:.2f}")
print(f"frequency = {freq_t:.2f}, pure = ${pure_t:,.0f}, gross = ${gross_t:,.2f}")
print(f"saving vs age-based premium: ${saving:,.2f} ({saving/2545.71:.1%})")

telematics multiplier = e^(-0.326) = 0.722; relativity 1.8 -> 1.30
frequency = 0.13, pure = $1,287, gross = $1,838.57
saving vs age-based premium: $707.14 (27.8%)


## 8. Claims triage logistic scores (Section 11.3.5)

In [8]:
def logistic(z):
    return 1/(1 + math.exp(-z))

intercept, b_attorney, b_injury, b_late = -2.2, 1.6, 1.3, 0.8
claim_A = logistic(intercept + b_attorney + b_injury)   # attorney + injury, prompt
claim_B = logistic(intercept + b_late)                  # late report only
print(f"claim A (attorney+injury): P(> $50k) = {claim_A:.4f} -> senior adjuster")
print(f"claim B (late report):     P(> $50k) = {claim_B:.4f} -> straight-through processing")

claim A (attorney+injury): P(> $50k) = 0.6682 -> senior adjuster
claim B (late report):     P(> $50k) = 0.1978 -> straight-through processing


## 9. Chain-ladder reserving (Section 11.4.1)

In [9]:
f_12_24 = (600+675)/(400+450)
f_24_36 = 660/600
ult2 = 675*f_24_36
ult3 = 500*f_12_24*f_24_36
reserve = (ult2-675) + (ult3-500)
print(f"development factors: {f_12_24:.2f} (12->24), {f_24_36:.2f} (24->36)")
print(f"ultimates: Year 2 = {ult2:.1f}, Year 3 = {ult3:.1f}; total reserve = ${reserve:.1f} thousand")

development factors: 1.50 (12->24), 1.10 (24->36)
ultimates: Year 2 = 742.5, Year 3 = 825.0; total reserve = $392.5 thousand


## 10. Term-life net premium (Section 11.4.2)

In [10]:
q, B, r = 0.002, 500_000, 0.04
net_premium = q*B/(1+r)
print(f"one-year term net premium = ${net_premium:,.2f}")

one-year term net premium = $961.54


## 11. Risk adjustment: the HCC score and an upcoding audit (Section 11.4.3)

In [11]:
components = {'demographic (76F)': 0.45, 'diabetes w/ complications': 0.30,
              'heart failure': 0.33, 'COPD': 0.22}
score = sum(components.values())
base_payment = 10_000
print(f"risk score = {score:.2f} -> payment ${base_payment*score:,.0f}")
upcoded = score + 0.13
print(f"with one unsupported vascular code (+0.13): {upcoded:.2f} -> ${base_payment*upcoded:,.0f} "
      f"(+${base_payment*0.13:,.0f}/enrollee/yr)")

# audit z-test: coded prevalence vs clinical benchmark
p0, p_hat, n_enroll = 0.06, 0.09, 10_000
z_audit = (p_hat - p0)/math.sqrt(p0*(1-p0)/n_enroll)
print(f"audit: coded prevalence {p_hat:.0%} vs benchmark {p0:.0%} on n={n_enroll:,} -> z = {z_audit:.1f}")

risk score = 1.30 -> payment $13,000
with one unsupported vascular code (+0.13): 1.43 -> $14,300 (+$1,300/enrollee/yr)
audit: coded prevalence 9% vs benchmark 6% on n=10,000 -> z = 12.6


## 12. ALM with dynamic lapse (Section 11.4.4)

In [12]:
A, D_A, L = 110.0, 5, 100.0
surplus0 = A - L
for label, dy, D_L in [("naive, rates +1%", +0.01, 12), ("naive, rates -1%", -0.01, 12),
                        ("dynamic, rates +1% (lapses accelerate, D_L 12->9)", +0.01, 9),
                        ("dynamic, rates -1% (lapses slow, D_L 12->14)", -0.01, 14)]:
    dA = -A*D_A*dy
    dL = -L*D_L*dy
    print(f"{label:52s}: surplus {surplus0:.0f} -> {A+dA-(L+dL):.1f}")

# the dynamic-lapse function of Exercise 8
for r in (0.03, 0.05, 0.06):
    lapse = 0.05 + 4*max(0.0, r-0.03)
    print(f"lapse rate at market rate {r:.0%}: {lapse:.0%}")

naive, rates +1%                                    : surplus 10 -> 16.5
naive, rates -1%                                    : surplus 10 -> 3.5
dynamic, rates +1% (lapses accelerate, D_L 12->9)   : surplus 10 -> 13.5
dynamic, rates -1% (lapses slow, D_L 12->14)        : surplus 10 -> 1.5
lapse rate at market rate 3%: 5%
lapse rate at market rate 5%: 13%
lapse rate at market rate 6%: 17%


## 13. Reinsurance layer and parametric basis risk (Section 11.5.1)

In [13]:
# layer: $100M xs $50M
p_attach, e_loss_in_layer, limit = 0.04, 60.0, 100.0
expected_loss = p_attach*e_loss_in_layer
print(f"layer expected annual loss = ${expected_loss:.1f}M; rate on line = {expected_loss/limit:.1%}")

# parametric policy
p_dmg, p_hit, p_false = 0.05, 0.80, 0.01
p_miss = p_dmg*(1-p_hit)                 # damage, no payout
p_false_pay = (1-p_dmg)*p_false          # payout, no damage
p_pay = p_dmg*p_hit + p_false_pay
B_param = 200_000
print(f"P(damage, no payout) = {p_miss:.4f}; P(payout, no damage) = {p_false_pay:.4f}")
print(f"P(payout) = {p_pay:.4f}; pure premium = ${p_pay*B_param:,.0f}")

layer expected annual loss = $2.4M; rate on line = 2.4%
P(damage, no payout) = 0.0100; P(payout, no damage) = 0.0095
P(payout) = 0.0495; pure premium = $9,900


## 14. Solvency capital as a 99.5% VaR (Section 11.5.2)

In [14]:
sd_nav = 40.0   # $M, one-year net-asset-value standard deviation
z995 = NormalDist().inv_cdf(0.995)
print(f"z(99.5%) = {z995:.3f}; SCR (normal approx) = {z995*sd_nav:.1f}M")
print("(underwriting results are compound-Poisson skewed, so the normal approximation understates the tail)")

z(99.5%) = 2.576; SCR (normal approx) = 103.0M
(underwriting results are compound-Poisson skewed, so the normal approximation understates the tail)


## Exercises (match Chapter 11, Suggested Exercises)

Selected exercises reproduced below; use the cells above as templates for the others.

In [15]:
# Exercise 2: experienced rural driver
freq_ex = 0.08*0.9*0.8
sev_ex = 9_000*0.95
gross_ex = freq_ex*sev_ex/0.70
print(f"Exercise 2 -- gross premium = ${gross_ex:,.2f} ({gross_ex/2545.71:.1%} of the young urban driver's)")

# Exercise 5: five-year fleet credibility
Z5 = 5/(5+12)
blend5 = Z5*0.30 + (1-Z5)*0.10
n75 = 3*12   # Z = n/(n+k) = 0.75 -> n = 3k
print(f"Exercise 5 -- Z = {Z5:.3f}, blend = {blend5:.4f}; years for 75% weight = {n75}")

# Exercise 6: fourth accident year
ult4 = 550*(600+675)/(400+450)*(660/600)
print(f"Exercise 6 -- Year 4 ultimate = {ult4:.1f}, added reserve = {ult4-550:.1f}")

# Exercise 7: hedged ALM (liability duration 8)
A, L = 110.0, 100.0
print(f"Exercise 7 -- hedged, rates -1%: surplus -> {A + A*5*0.01 - (L + L*8*0.01):.1f}; "
      f"rates +1%: surplus -> {A - A*5*0.01 - (L - L*8*0.01):.1f}")

# Exercise 9: higher layer
print(f"Exercise 9 -- expected loss = {0.02*30:.1f}M, rate on line = {0.02*30/50:.1%}")

# Exercise 10: better parametric index
p_miss2 = 0.05*(1-0.95); p_false2 = 0.95*0.01
p_pay2 = 0.05*0.95 + p_false2
print(f"Exercise 10 -- P(damage,no payout) = {p_miss2:.4f}, P(payout,no damage) = {p_false2:.4f}, "
      f"pure premium = ${p_pay2*200_000:,.0f}")

# Exercise 11: small-book audit z
z_small = (0.07-0.06)/math.sqrt(0.06*0.94/2500)
print(f"Exercise 11 -- z = {z_small:.2f} (same direction, far weaker evidence)")

Exercise 2 -- gross premium = $703.54 (27.6% of the young urban driver's)
Exercise 5 -- Z = 0.294, blend = 0.1588; years for 75% weight = 36
Exercise 6 -- Year 4 ultimate = 907.5, added reserve = 357.5
Exercise 7 -- hedged, rates -1%: surplus -> 7.5; rates +1%: surplus -> 12.5
Exercise 9 -- expected loss = 0.6M, rate on line = 1.2%
Exercise 10 -- P(damage,no payout) = 0.0025, P(payout,no damage) = 0.0095, pure premium = $11,400
Exercise 11 -- z = 2.11 (same direction, far weaker evidence)
